# EIA Weekly Petroleum Inventories

This notebook downloads weekly U.S. commercial crude oil inventory data from the
Energy Information Administration (EIA) public API. Each observation corresponds
to the Weekly Petroleum Status Report published every Wednesday at 10:30 AM ET.
The notebook computes the week-on-week inventory change and classifies each report
as bearish (inventories increased) or bullish (inventories decreased), producing
structured events with exact timestamps, then aligns them into `market_context`.

Results are written to:
- **DB:** `eia_events` and `market_context` tables in `01_data/wti_thesis.db` (primary)
- **CSV backup:** `01_data/raw/macro/eia_inventories_raw.csv`

### General imports

In [1]:
import requests
import pandas as pd
import sqlite3
import os

### Download data from EIA

EIA allows to download weekly oil production and inventory reports

In [3]:
url = "https://api.eia.gov/v2/petroleum/stoc/wstk/data/"
params = {
    "api_key": "DEMO_KEY",
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "WCRSTUS1",
    "start": "2020-01-01",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": 0,
    "length": 5000
}
response = requests.get(url, params=params)
df_eia = pd.DataFrame(response.json()['response']['data'])

### Clean data

- **Column selection:** Only `period` (the date) and `value` (inventory level in thousand barrels) are retained.
- **Type conversion:** `value` is a string in the API response and is coerced to numeric.
- **Weekly shock:** `inventory_change = value_this_week − value_previous_week`. The market reacts to the deviation from the prior week, not the absolute level.
- **Event direction:** `bearish` = inventories up (oversupply), `bullish` = inventories down (scarcity).
- **Timestamp construction:** The EIA API `period` field for WCRSTUS1 is the **Friday** of the reporting week (the data collection end date). The actual report is published **2 days earlier on Wednesday** at 10:30 AM ET. Subtracting 2 days before adding 10:30 converts Friday → Wednesday, then DST-aware tz_localize + tz_convert produces the correct UTC release timestamp. `ceil('h')` rounds 10:30 → 11:00 local → the exact UTC hour the market first sees the data.

In [4]:
df_eia = df_eia[['period', 'value']].copy()
df_eia.columns = ['date', 'inventory_mbbl']
df_eia['date'] = pd.to_datetime(df_eia['date'])
df_eia['inventory_mbbl'] = pd.to_numeric(df_eia['inventory_mbbl'], errors='coerce')
df_eia = df_eia.sort_values('date').reset_index(drop=True)

df_eia['inventory_change'] = df_eia['inventory_mbbl'].diff()
df_eia['news_direction'] = df_eia['inventory_change'].apply(
    lambda x: 'bearish' if x > 0 else ('bullish' if x < 0 else 'neutral')
)

# `period` is the Friday week-end date; EIA publishes 2 days earlier on Wednesday at 10:30 AM ET
df_eia['datetime_et'] = pd.to_datetime(df_eia['date']) - pd.Timedelta(days=2) + pd.Timedelta(hours=10, minutes=30)
df_eia['datetime_et'] = df_eia['datetime_et'].dt.tz_localize('US/Eastern')
df_eia['datetime_et'] = df_eia['datetime_et'].dt.tz_convert('UTC')
df_eia['datetime_et'] = df_eia['datetime_et'].dt.ceil('h')

print(f"Entries: {len(df_eia)}")
print(f"Range: {df_eia['date'].min().date()} to {df_eia['date'].max().date()}")
print(f"Bearish: {(df_eia['news_direction']=='bearish').sum()}")
print(f"Bullish: {(df_eia['news_direction']=='bullish').sum()}")

# Verify releases fall on Wednesdays (weekday=2)
days = df_eia['datetime_et'].dt.day_name().value_counts()
print(f"\nRelease day distribution (should be Wednesday):\n{days}")
print(df_eia[['date', 'inventory_change', 'news_direction', 'datetime_et']].tail(5))

Entries: 331
Range: 2020-01-03 to 2026-05-01
Bearish: 139
Bullish: 191

Release day distribution (should be Wednesday):
datetime_et
Wednesday    331
Name: count, dtype: int64
          date  inventory_change news_direction               datetime_et
326 2026-04-03            1342.0        bearish 2026-04-01 15:00:00+00:00
327 2026-04-10           -5057.0        bullish 2026-04-08 15:00:00+00:00
328 2026-04-17           -2211.0        bullish 2026-04-15 15:00:00+00:00
329 2026-04-24          -13355.0        bullish 2026-04-22 15:00:00+00:00
330 2026-05-01           -7537.0        bullish 2026-04-29 15:00:00+00:00


### Save to DB

In [5]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_eia['date'] = df_eia['date'].astype(str)
df_eia['datetime_et'] = df_eia['datetime_et'].astype(str)

df_eia.to_sql('eia_events', conn, if_exists='replace', index=False)
print(f"EIA records saved to DB: {len(df_eia)}")

# CSV backup
raw_path = "../01_data/raw/macro/eia_inventories_raw.csv"
os.makedirs(os.path.dirname(os.path.abspath(raw_path)), exist_ok=True)
df_eia.to_csv(raw_path, index=False)
print(f"CSV backup: {raw_path}")

# Verify
print(pd.read_sql(
    "SELECT MIN(date) as earliest, MAX(date) as latest, COUNT(*) as total FROM eia_events",
    conn
))
print(pd.read_sql("SELECT date, datetime_et FROM eia_events LIMIT 5", conn))
conn.close()

EIA records saved to DB: 331
CSV backup: ../01_data/raw/macro/eia_inventories_raw.csv
     earliest      latest  total
0  2020-01-03  2026-05-01    331
         date                datetime_et
0  2020-01-03  2020-01-01 16:00:00+00:00
1  2020-01-10  2020-01-08 16:00:00+00:00
2  2020-01-17  2020-01-15 16:00:00+00:00
3  2020-01-24  2020-01-22 16:00:00+00:00
4  2020-01-31  2020-01-29 16:00:00+00:00


### Merge EIA entries into market_context

In [ ]:
conn = sqlite3.connect("../01_data/wti_thesis.db")

df_eia = pd.read_sql("SELECT inventory_change, datetime_et FROM eia_events", conn)
df_eia['datetime_et'] = pd.to_datetime(df_eia['datetime_et'], utc=True)
df_eia = df_eia.sort_values('datetime_et').reset_index(drop=True)

df_market = pd.read_sql("SELECT * FROM market_context", conn)
df_market['datetime_hour'] = pd.to_datetime(df_market['datetime_hour'], utc=True)
df_market = df_market.sort_values('datetime_hour').reset_index(drop=True)

df_merged = pd.merge_asof(
    df_market,
    df_eia[['datetime_et', 'inventory_change']],
    left_on='datetime_hour',
    right_on='datetime_et',
    direction='backward'
)

df_merged = df_merged.rename(columns={'inventory_change': 'eia_surprise'})
df_merged['eia_surprise'] = df_merged['eia_surprise'].fillna(0)
df_merged['is_eia_release'] = (df_merged['datetime_et'] == df_merged['datetime_hour']).astype(int)
df_merged = df_merged.drop(columns=['datetime_et'])

df_merged['datetime_hour'] = df_merged['datetime_hour'].astype(str)
df_merged.to_sql('market_context', conn, if_exists='replace', index=False)
print(f"market_context updated: {len(df_merged)} rows")

# Verify: is_eia_release=1 should fall on Wednesdays
releases = pd.read_sql(
    "SELECT datetime_hour, eia_surprise, is_eia_release FROM market_context WHERE is_eia_release = 1 LIMIT 10",
    conn
)
releases['day'] = pd.to_datetime(releases['datetime_hour']).dt.day_name()
print("\nEIA release hours in market_context (should all be Wednesday):")
print(releases[['datetime_hour', 'eia_surprise', 'day']].to_string())

# Spot-check: the week of 2024-05-15 (a Wednesday)
print("\nSpot-check 2024-05-14 to 2024-05-17:")
print(pd.read_sql("""
    SELECT datetime_hour, eia_surprise, is_eia_release
    FROM market_context
    WHERE datetime_hour >= '2024-05-14' AND datetime_hour <= '2024-05-16'
    AND strftime('%H', datetime_hour) = '15'
    ORDER BY datetime_hour
""", conn))

# Save market_context CSV backup with EIA columns included
mc_path = "../01_data/processed/market_context.csv"
df_merged.to_csv(mc_path, index=False)
print(f"\nmarket_context CSV backup updated: {mc_path}")

conn.close()

OperationalError: duplicate column name: eia_surprise

: 